# 15 — Roboflow Inference SDK con WebRTC

*Duración estimada: 1 hora*

## 1. El desafío del Despliegue en Tiempo Real

Crear un cuaderno de Jupyter que detecte objetos es fácil. Llevar ese modelo a un producto real, en vivo, con una cámara de un usuario al otro lado del mundo, es un reto gigantesco de ingeniería de software.

Si usamos el clásico protocolo HTTP (API REST), enviar un frame, esperar a que el servidor lo procese y lo devuelva, genera una latencia de 1 a 2 segundos por frame. Esto hace que las aplicaciones de video en vivo (como filtros de Instagram, asistencia por video, o robótica remota) sean imposibles.

**La solución:** WebRTC.

## 2. ¿Qué es WebRTC?

Web Real-Time Communication (WebRTC) es una tecnología de código abierto que permite la comunicación de video, voz y datos en tiempo real entre navegadores o dispositivos, **sin plugins**.

- Funciona sobre UDP (más rápido que TCP porque no espera confirmación de recibo de paquetes).
- Latencia ultra-baja (< 100ms).
- Es peer-to-peer (P2P), aunque para inferencia de IA lo conectamos a un servidor intermedio (Media Server).

## 3. Roboflow Inference SDK

Roboflow ofrece un paquete en Python llamado `inference` que actúa como un servidor ultra-optimizado para correr modelos YOLO, SAM y otros, aprovechando la aceleración de hardware (ONNX, TensorRT, CoreML).

In [1]:
!pip install -q inference aiortc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 10.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 

## 4. Arquitectura del Pipeline WebRTC + Inferencia

1. El **Navegador web (Cliente)** captura la webcam y envía el track de video por WebRTC al Servidor.
2. El **Servidor (Python con aiortc)** intercepta los frames de video uno por uno.
3. El Servidor envía el frame a **Roboflow Inference SDK** alojado localmente.
4. El modelo retorna las cajas y máscaras.
5. El Servidor usa `supervision` o OpenCV para dibujar las máscaras sobre el frame original.
6. El Servidor devuelve el track de video modificado al navegador web, todo en <50 milisegundos.

In [ ]:
# ESQUELETO CONCEPTUAL DE SERVIDOR WebRTC EN PYTHON
# (Nota: Este código es conceptual para entender la arquitectura.
# Un servidor WebRTC real requiere configuración de SDP y señalización).

'''
import cv2
from aiortc import VideoStreamTrack
from inference import get_model
import supervision as sv

# 1. Cargar el modelo ultra-rápido desde Roboflow (ej. YOLOv8 Nano para Segmentación)
model = get_model(model_id="my-segmentation-project/1")

# 2. Configurar Supervision
mask_annotator = sv.MaskAnnotator()

# 3. Crear una clase que procese el stream de video de WebRTC
class AIProcessingTrack(VideoStreamTrack):
    def __init__(self, track_entrada):
        super().__init__()
        self.track_entrada = track_entrada

    async def recv(self):
        # 3.1 Recibir frame del usuario desde la web
        frame_obj = await self.track_entrada.recv()

        # 3.2 Convertir a formato Numpy para OpenCV
        img = frame_obj.to_ndarray(format="bgr24")

        # 3.3 Inferencia ultra-rápida
        results = model.infer(img)

        # 3.4 Convertir a formato Supervision (Asumiendo adaptador previo)
        detections = sv.Detections.from_roboflow(results)

        # 3.5 Dibujar sobre la imagen
        img_dibujada = mask_annotator.annotate(scene=img.copy(), detections=detections)

        # 3.6 Re-empaquetar y devolver al navegador web
        nuevo_frame = VideoFrame.from_ndarray(img_dibujada, format="bgr24")
        nuevo_frame.pts = frame_obj.pts
        nuevo_frame.time_base = frame_obj.time_base
        return nuevo_frame
'''
print("Arquitectura conceptual cargada.")

## 5. Ejercicio de Arquitectura

Si fueras a desplegar un modelo SAM3 (que requiere GPU) para una aplicación móvil donde el usuario usa la cámara en vivo:
1. ¿Dónde alojarías el servidor de inferencia?
2. ¿Cómo manejarías la caída de frames si la red del usuario es lenta?

*Discutiremos esto al final de la clase.*

In [2]:
%pip install -q pydantic==2.6.4 fastapi==0.112.2 gradio==4.44.1 fastrtc==0.0.25 huggingface_hub==0.25.2 ultralytics supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.5/93.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 163.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5

In [1]:
import cv2
import gradio as gr
from ultralytics import YOLO
import supervision as sv

# 1. Cargamos el modelo
model = YOLO("yolov8n.pt")
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

# 2. Función de procesamiento (recibe el frame por WebSocket)
def procesar_frame(frame):
    if frame is None:
        return None

    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    results = model(frame_bgr, conf=0.15, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    labels = [f"{model.names[c]} {conf:.2f}" for c, conf in zip(detections.class_id, detections.confidence)]

    annotated_bgr = box_annotator.annotate(scene=frame_bgr.copy(), detections=detections)
    annotated_bgr = label_annotator.annotate(scene=annotated_bgr, detections=detections, labels=labels)
    cv2.circle(annotated_bgr, (50, 50), 30, (0, 255, 0), -1)

    return cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

# 3. Interfaz gráfica con CSS para BORRAR el logo que te estorba
css = """
footer {display: none !important;}
"""

with gr.Blocks(css=css) as demo:
    gr.Markdown("# 📷 YOLOv8 Video en Vivo (Solución WebSockets Colab)")

    with gr.Row():
        entrada_video = gr.Image(sources=["webcam"], streaming=True)
        salida_video = gr.Image()

    # Conectamos la entrada con la salida
    entrada_video.stream(fn=procesar_frame, inputs=entrada_video, outputs=salida_video)

# 4. Lanzamos la app con share=True
demo.launch(share=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://2de490f637c2f959e7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
from fastrtc import Stream
import numpy as np


def flip_vertically(image):
    return np.flip(image, axis=0)
    # return image


stream = Stream(
    handler=flip_vertically,
    modality="video",
    mode="send-receive",
)

stream.ui.launch()